# 🧪 Lab 02: The Cost of Being Generic

Now that we can recognize a codegen stage, we can ask the more interesting question: why did Spark build this machinery at all?

**Mission Objective:** run the same deliberately CPU-heavy workload with WholeStageCodegen enabled and disabled. We will verify that the physical execution path changes, force the computed value into a checksum so Catalyst cannot prune it, and compare repeated warm runs.

**Deterministic Guardrail:** both variants use the same generated input, expression, partition count, and final aggregation. This is a controlled diagnostic, not a universal claim that codegen is always faster.


### Step 1: Define the diagnostic session
The session is local and small enough to run from the notebook, but the workload is large enough to repeat the expression many times. The Spark UI is disabled because this lab compares execution paths, not UI telemetry.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = (SparkSession.builder
    .master("local[2]")
    .appName("lab-02-make-codegen-pay")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("WholeStageCodegen default:", spark.conf.get("spark.sql.codegen.wholeStage"))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:22:08 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:22:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/24 06:22:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
WholeStageCodegen default: true


### Step 2: Build a workload that cannot discard the expression
A plain `select(...).count()` could allow Catalyst to remove the projection because the count only needs row cardinality. Here the expensive expression feeds `sum`, so its value is required to produce the final checksum.


In [2]:
def build_workload():
    return (spark.range(0, 2_000_000, 1, numPartitions=2)
        .where((F.col("id") % 7) == 0)
        .select((
            F.sqrt(F.col("id") + 1.0)
            + F.log1p(F.col("id") + 1.0)
            + F.sin(F.col("id") * 0.001)
            + F.cos(F.col("id") * 0.002)
        ).alias("work"))
        .agg(F.sum("work").alias("checksum")))

def run_once(enabled):
    spark.conf.set("spark.sql.codegen.wholeStage", str(enabled).lower())
    query = build_workload()
    start = time.perf_counter()
    checksum = query.collect()[0]["checksum"]
    elapsed = time.perf_counter() - start
    return enabled, checksum, elapsed


### Step 3: Confirm that the execution path changes
We inspect equivalent plans with the setting enabled and disabled. The output should show `*(n)` markers when WholeStageCodegen is enabled and an unstarred physical plan when it is disabled.


In [3]:
spark.conf.set("spark.sql.codegen.wholeStage", "true")
print("=== WholeStageCodegen enabled ===")
build_workload().explain("formatted")

spark.conf.set("spark.sql.codegen.wholeStage", "false")
print("=== WholeStageCodegen disabled ===")
build_workload().explain("formatted")


=== WholeStageCodegen enabled ===


== Physical Plan ==
* HashAggregate (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- * Project (3)
         +- * Filter (2)
            +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : ((id#0L % 7) = 0)

(3) Project [codegen id : 1]
Output [1]: [(((SQRT((cast(id#0L as double) + 1.0)) + LOG1P((cast(id#0L as double) + 1.0))) + SIN((cast(id#0L as double) * 0.001))) + COS((cast(id#0L as double) * 0.002))) AS work#4]
Input [1]: [id#0L]

(4) HashAggregate [codegen id : 1]
Input [1]: [work#4]
Keys: []
Functions [1]: [partial_sum(work#4)]
Aggregate Attributes [1]: [sum#8]
Results [1]: [sum#9]

(5) Exchange
Input [1]: [sum#9]
Arguments: SinglePartition, ENSURE_REQUIREMENTS, [plan_id=21]

(6) HashAggregate [codegen id : 2]
Input [1]: [sum#9]
Keys: []
Functions [1]: [sum(work#4)]
Aggregate Attributes [1]: [sum(work#4)#7]
Results [1]: [sum(work#4)#7 AS checksum#5]


### Step 4: Warm up and measure both execution paths
The first actions can include general JVM/Spark warm-up as well as code-generation work, so we discard them rather than trying to attribute that cost precisely. We then alternate both execution paths to reduce measurement-order bias. The checksum is printed for every run so both variants are held to the same result.


In [4]:
# Warm both execution paths once. These timings are intentionally ignored.
for setting in (True, False):
    warm = run_once(setting)
    print(
        f"warm-up codegen={setting}: "
        f"checksum={warm[1]:.6f}, elapsed={warm[2]:.3f}s"
    )

# Alternate measured runs to reduce measurement-order bias.
order = [True, False, False, True, True, False]
results = []
run_counts = {True: 0, False: 0}

for sequence_number, setting in enumerate(order, 1):
    result = run_once(setting)
    run_counts[setting] += 1
    results.append((setting, run_counts[setting], result[1], result[2]))

    print(
        f"sequence={sequence_number}, measured run={run_counts[setting]}, "
        f"codegen={setting}: checksum={result[1]:.6f}, elapsed={result[2]:.3f}s"
    )

for setting in (True, False):
    timings = [row[3] for row in results if row[0] == setting]
    print(
        f"summary codegen={setting}: "
        f"mean={sum(timings) / len(timings):.3f}s, "
        f"min={min(timings):.3f}s"
    )

checksums = {round(row[2], 6) for row in results}
print("distinct measured checksums:", checksums)


warm-up codegen=True: checksum=273234288.125562, elapsed=1.791s


warm-up codegen=False: checksum=273234288.125562, elapsed=0.683s


sequence=1, measured run=1, codegen=True: checksum=273234288.125562, elapsed=0.176s


sequence=2, measured run=1, codegen=False: checksum=273234288.125562, elapsed=0.300s


sequence=3, measured run=2, codegen=False: checksum=273234288.125562, elapsed=0.166s
sequence=4, measured run=2, codegen=True: checksum=273234288.125562, elapsed=0.104s


sequence=5, measured run=3, codegen=True: checksum=273234288.125562, elapsed=0.107s


sequence=6, measured run=3, codegen=False: checksum=273234288.125562, elapsed=0.144s
summary codegen=True: mean=0.129s, min=0.104s
summary codegen=False: mean=0.203s, min=0.144s
distinct measured checksums: {273234288.125562}


# 📊 Post-Lab Analysis: The Cost of Being Generic

This lab held the input, expression, checksum, and partitioning constant while changing only the WholeStageCodegen setting. The physical plans confirmed that the setting changes the execution path, and the repeated measurements showed what that change cost for this workload.

### 1. The Expression Was Actually Paid For

Because the computed column fed `sum`, Catalyst could not satisfy the action by counting rows and discarding the projection. The matching checksums are the correctness guardrail: both paths produced the same required result.

### 2. Generic Execution Has Repetition as Its Enemy

The comparison isolates WholeStageCodegen, not all Spark code generation. With WholeStageCodegen disabled, Spark loses the fused stage-level execution path, but other forms of expression/operator code generation may still exist. The measured difference therefore tells us what whole-stage fusion contributed to this workload—not the cost of every individual abstraction inside Spark.

### 3. Code Generation Has a Price

The warm-up runs are discarded because they can mix JVM/Spark startup, class loading, and code-generation work. The measured runs then alternate both settings to reduce order bias and focus the comparison on steady-state behavior. Generated code still has setup cost; this lab simply does not pretend to isolate that cost precisely.

### 4. What This Experiment Proves

This run supports a workload-specific conclusion: if the codegen variant is faster after warm-up, repeated local execution overhead mattered here. If the timings are close, codegen was not the dominant cost. Neither result makes a universal claim about every Spark query.
